In [1]:
import sys
sys.path.append("/axovol/l1ad/src")  # project root
### Imports
import json
import numpy as np
from torch.utils import data
import yaml
import h5py
import torch
import trainer
import model


2026-04-03 20:47:15.314965: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-03 20:47:15.358646: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-03 20:47:15.358694: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-03 20:47:15.358726: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-03 20:47:15.366989: I tensorflow/core/platform/cpu_feature_g

In [2]:
def load_config(config_path, overrides=None):
    """Load YAML config and apply CLI overrides."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    if overrides:
        for key, value in overrides.items():
            keys = key.split(".")
            sub = config
            for k in keys[:-1]:
                sub = sub.setdefault(k, {})
            sub[keys[-1]] = value
    return config

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
device

device(type='cuda')

In [5]:
f = h5py.File('../training/v5/conditionsupdate_apr25.h5', 'r')

In [6]:
x_train = f['data']["Background_data"]["Train"]["DATA"][:50000]
x_test = f['data']["Background_data"]["Test"]["DATA"][:50000]
x_sig = f['data']["Signal_data"]["GluGluHToBB_M-125"]["DATA"][:50000]

scale = f['data']["Normalisation"]["norm_scale"][:]
bias = f['data']["Normalisation"]["norm_bias"][:]

In [7]:
x_train = torch.tensor(np.reshape(x_train,(x_train.shape[0],-1))).to(torch.float32).to(device)
x_test = torch.tensor(np.reshape(x_test,(x_test.shape[0],-1))).to(torch.float32).to(device)
x_sig = torch.tensor(np.reshape(x_sig,(x_sig.shape[0],-1))).to(torch.float32).to(device)


In [8]:
vicreg_encoder = model.Encoder(
    input_size=57,
    intermediate_architecture=[29],
    bottleneck_size=10,
    drop_out=None
).to(device)

vicreg_encoder.load_state_dict(torch.load("encoder.pth"))
vicreg_encoder.eval()

with torch.no_grad():
    x_train = vicreg_encoder(x_train)
    x_test  = vicreg_encoder(x_test)
    x_sig   = vicreg_encoder(x_sig)

In [9]:
batch_size = 2048

train_loader = data.DataLoader(
    dataset=data.TensorDataset(x_train),
    batch_size=batch_size,
)

val_loader = data.DataLoader(
    dataset=data.TensorDataset(x_test),
    batch_size=batch_size,
)

val_loader_no_batch = data.DataLoader(
    dataset=data.TensorDataset(x_test),
    batch_size=len(x_test),  # was len(x_train), should be len(x_test)
)

sig_loader = data.DataLoader(
    dataset=data.TensorDataset(x_sig),
    batch_size=batch_size,
)

class MyLoader():
    def __init__(self, train_loader, val_loader, val_loader_no_batch, ood_loader) -> None:
        self.training_loader = train_loader
        self.validation_loader = val_loader
        self.validation_loader_no_batch = val_loader_no_batch
        self.ood_loader = ood_loader
        
loaders = MyLoader(train_loader, val_loader, val_loader_no_batch, sig_loader)

In [10]:
config = yaml.safe_load(open("config/config.yaml", "r"))
config['training']["batch_size"] = batch_size
config['training']['n_epochs'] = 5

In [11]:
config

{'data': {'filepath': '../training/v5/conditionsupdate_apr25.h5',
  'output': 'output_full_10_21',
  'standardize': False,
  'min_max': True,
  'n_train_sample': 50000,
  'n_test_sample': 10000},
 'training': {'batch_size': 2048,
  'es_patience': 5000,
  'n_epochs': 5,
  'optimizer': 'AdamW',
  'learning_rate': 0.005,
  'lr_scheduler': 'ReduceLROnPlateau',
  'lr_scheduler_args': {'mode': 'min',
   'factor': 0.8,
   'patience': 20,
   'threshold': 0.001,
   'cooldown': 10,
   'threshold_mode': 'rel',
   'min_lr': 0}},
 'wnae': {'sampling': 'pcd',
  'x_step': 5,
  'x_step_size': 0.05,
  'x_noise_std': 0.22,
  'x_temperature': 0.063,
  'x_bound': [-3, 3],
  'x_clip_grad': None,
  'x_reject_boundary': False,
  'x_mh': False,
  'z_step': 5,
  'z_step_size': 0.05,
  'z_temperature': 0.063,
  'z_noise_std': 1,
  'z_bound': None,
  'z_clip_grad': None,
  'z_reject_boundary': False,
  'z_mh': False,
  'spherical': False,
  'initial_dist': 'gaussian',
  'replay': True,
  'replay_ratio': 0.95,
  

In [19]:
input_size = x_train.shape[-1]  # will be 8 after vicreg transform

encoder = model.VAE_Encoder(
    input_size=input_size,
    intermediate_architecture=[6],
    bottleneck_size=4,
    drop_out=None
)

decoder = model.Decoder(
    output_size=input_size,
    intermediate_architecture=[6],
    bottleneck_size=4,
    drop_out=None
)

encoder = encoder.to(device)
decoder = decoder.to(device)

vae = model.VariationalAutoEncoder(encoder=encoder, decoder=decoder).to(device)

optimizer = torch.optim.AdamW(vae.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.8, patience=20)

history = {
    'total_loss': [], 'reco_loss': [], 'kl_loss': [],
    'val_total_loss': [], 'val_reco_loss': [], 'val_kl_loss': []
}

In [20]:
import os
import matplotlib.pyplot as plt

vae = model.VariationalAutoEncoder(encoder=encoder, decoder=decoder).to(device)

optimizer_class = getattr(torch.optim, config['training']['optimizer'])
optimizer = optimizer_class(vae.parameters(), lr=config['training']['learning_rate'])

scheduler_class = getattr(torch.optim.lr_scheduler, config['training']['lr_scheduler'])
scheduler = scheduler_class(optimizer, **config['training']['lr_scheduler_args'])

history = {
    'total_loss': [], 'reco_loss': [], 'kl_loss': [],
    'val_total_loss': [], 'val_reco_loss': [], 'val_kl_loss': []
}

In [21]:
config['training']['optimizer']

'AdamW'

In [22]:
config['training']['n_epochs'] = 200

In [23]:
for epoch in range(config['training']['n_epochs']):

    # KL annealing - linearly increase from 0 to 1 over first 50 epochs
    kl_weight = min(1.0, epoch / 50)

    vae.train()
    total_loss_sum = 0
    reco_loss_sum  = 0
    kl_loss_sum    = 0
    batch_count    = 0

    for (batch,) in loaders.training_loader:

        batch = batch.to(device)

        x_hat, mu, log_var = vae(batch)
        total_loss, reco_loss, kl_loss = model.VariationalAutoEncoder.loss(
            batch, x_hat, mu, log_var
        )

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        total_loss_sum += total_loss.item()
        reco_loss_sum  += reco_loss.item()
        kl_loss_sum    += kl_loss.item()
        batch_count    += 1

    avg_total = total_loss_sum / batch_count
    avg_reco  = reco_loss_sum  / batch_count
    avg_kl    = kl_loss_sum    / batch_count

    # -------------------------
    # Validation
    # -------------------------

    vae.eval()

    val_total_sum = 0
    val_reco_sum  = 0
    val_kl_sum    = 0
    val_count     = 0

    with torch.no_grad():
        for (batch,) in loaders.validation_loader:

            batch = batch.to(device)

            x_hat, mu, log_var = vae(batch)
            val_loss, val_reco, val_kl = model.VariationalAutoEncoder.loss(batch, x_hat, mu, log_var)

            val_total_sum += val_loss.item()
            val_reco_sum  += val_reco.item()
            val_kl_sum    += val_kl.item()
            val_count     += 1

    avg_val_total = val_total_sum / val_count
    avg_val_reco  = val_reco_sum  / val_count
    avg_val_kl    = val_kl_sum    / val_count

    scheduler.step(avg_val_total)

    # -------------------------
    # History
    # -------------------------

    history['total_loss'].append(avg_total)
    history['reco_loss'].append(avg_reco)
    history['kl_loss'].append(avg_kl)
    history['val_total_loss'].append(avg_val_total)
    history['val_reco_loss'].append(avg_val_reco)
    history['val_kl_loss'].append(avg_val_kl)

    # -------------------------
    # Print progress
    # -------------------------

    print(
        f"Epoch [{epoch+1}/{config['training']['n_epochs']}] | "
        f"Total: {avg_total:.4f} | Reco: {avg_reco:.4f} | KL: {avg_kl:.4f} | "
        f"Val Total: {avg_val_total:.4f} | Val Reco: {avg_val_reco:.4f} | Val KL: {avg_val_kl:.4f}"
    )

# -------------------------
# Plot
# -------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in zip(axes,
                           ['total_loss', 'reco_loss', 'kl_loss'],
                           ['Total Loss', 'Reconstruction Loss', 'KL Loss']):
    ax.plot(history[key], label='train')
    ax.plot(history[f'val_{key}'], label='val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()

plt.tight_layout()
os.makedirs(config['data']['output'], exist_ok=True)
plt.savefig(os.path.join(config['data']['output'], 'vae_training.png'))
plt.show()

Epoch [1/200] | Total: 0.2396 | Reco: 0.2156 | KL: 0.0240 | Val Total: 0.1549 | Val Reco: 0.1500 | Val KL: 0.0049
Epoch [2/200] | Total: 0.1259 | Reco: 0.1238 | KL: 0.0021 | Val Total: 0.1067 | Val Reco: 0.1053 | Val KL: 0.0014
Epoch [3/200] | Total: 0.0968 | Reco: 0.0958 | KL: 0.0009 | Val Total: 0.0901 | Val Reco: 0.0895 | Val KL: 0.0006
Epoch [4/200] | Total: 0.0854 | Reco: 0.0850 | KL: 0.0004 | Val Total: 0.0834 | Val Reco: 0.0831 | Val KL: 0.0003
Epoch [5/200] | Total: 0.0814 | Reco: 0.0812 | KL: 0.0003 | Val Total: 0.0814 | Val Reco: 0.0812 | Val KL: 0.0002
Epoch [6/200] | Total: 0.0801 | Reco: 0.0800 | KL: 0.0002 | Val Total: 0.0805 | Val Reco: 0.0804 | Val KL: 0.0001
Epoch [7/200] | Total: 0.0795 | Reco: 0.0794 | KL: 0.0001 | Val Total: 0.0802 | Val Reco: 0.0801 | Val KL: 0.0001
Epoch [8/200] | Total: 0.0793 | Reco: 0.0792 | KL: 0.0001 | Val Total: 0.0799 | Val Reco: 0.0799 | Val KL: 0.0001
Epoch [9/200] | Total: 0.0791 | Reco: 0.0790 | KL: 0.0001 | Val Total: 0.0798 | Val Reco

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7b707bade430>>
Traceback (most recent call last):
  File "/opt/conda/envs/axo/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


Epoch [17/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 | Val Reco: 0.0795 | Val KL: 0.0000
Epoch [18/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 | Val Reco: 0.0795 | Val KL: 0.0000
Epoch [19/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 | Val Reco: 0.0795 | Val KL: 0.0000
Epoch [20/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 | Val Reco: 0.0795 | Val KL: 0.0000
Epoch [21/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 | Val Reco: 0.0795 | Val KL: 0.0000
Epoch [22/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 | Val Reco: 0.0795 | Val KL: 0.0000
Epoch [23/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 | Val Reco: 0.0795 | Val KL: 0.0000
Epoch [24/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 | Val Reco: 0.0795 | Val KL: 0.0000
Epoch [25/200] | Total: 0.0787 | Reco: 0.0787 | KL: 0.0000 | Val Total: 0.0795 |

KeyboardInterrupt: 